In [1]:
import os
import sys
import shutil

# 1. Forcefully remove any existing cached modules from memory
for module in list(sys.modules.keys()):
    if any(pkg in module for pkg in ['transformers', 'trl', 'accelerate', 'peft','vllm']):
        del sys.modules[module]

# 2. Uninstall the system versions that are conflicting (the ones in /usr/local/lib)
!pip uninstall -y transformers accelerate trl peft bitsandbytes unsloth

# 3. Wipe your local lib folder to start fresh
if os.path.exists("/kaggle/working/lib"):
    shutil.rmtree("/kaggle/working/lib")
os.makedirs("/kaggle/working/lib")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 ── UNIVERSAL OFFLINE INSTALLATION (TAR.GZ METHOD)
# ═══════════════════════════════════════════════════════════════════════════════
import os
import sys
import subprocess
import shutil

# 1. Purge the memory cache to prevent Kaggle's base image from causing ghost conflicts
for module in list(sys.modules.keys()):
    if any(pkg in module for pkg in ['transformers', 'trl', 'accelerate', 'tokenizers', 'huggingface_hub', 'vllm']):
        del sys.modules[module]

# UPDATE THIS PATH to point to your new dataset containing the tar.gz
TAR_PATH = "/kaggle/input/notebooks/sachitbansal2006/qwen2-5-72b-utility/qwen35_offline_env.tar.gz" 
EXTRACT_DIR = "/kaggle/working/offline_env"
LOCAL_LIB = "/kaggle/working/lib"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
if os.path.exists(LOCAL_LIB):
    shutil.rmtree(LOCAL_LIB)

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(LOCAL_LIB, exist_ok=True)

# 2. Extract the archive
print(f"--- Extracting {TAR_PATH} ---")
subprocess.run(["tar", "-xzf", TAR_PATH, "-C", EXTRACT_DIR], check=True)

WHEELHOUSE = os.path.join(EXTRACT_DIR, "wheelhouse")
REQ_FILE = os.path.join(EXTRACT_DIR, "reqs.txt")

# 3. Let pip handle the installation automatically
print("--- Installing from local wheelhouse ---")
try:
    subprocess.run([
        "pip", "install", 
        "--no-index",                  # Do NOT look at the internet
        "--find-links", WHEELHOUSE,    # Look ONLY in our extracted folder
        "-r", REQ_FILE,                # Install everything listed here
        "--target", LOCAL_LIB,         # Keep it isolated from Kaggle's system files
        "--upgrade"                    # Overwrite Kaggle defaults if necessary
    ], check=True)
    print("\n✅ Installation Complete!")
except subprocess.CalledProcessError as e:
    print(f"\n❌ Installation Failed: {e}")

# 4. Path Injection & CUDA Fix
sys.path.insert(0, LOCAL_LIB)
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# 5. Verify Imports
try:
    import huggingface_hub
    import transformers
    import tokenizers
    import trl
    print(f"\n🚀 Success!")
    print(f"Transformers: {transformers.__version__}")
    print(f"Tokenizers: {tokenizers.__version__}")
    print(f"TRL: {trl.__version__}")
except ImportError as e:
    print(f"\n❌ FATAL IMPORT ERROR: {e}")

--- Extracting /kaggle/input/notebooks/sachitbansal2006/qwen2-5-72b-utility/qwen35_offline_env.tar.gz ---
--- Installing from local wheelhouse ---
Looking in links: /kaggle/working/offline_env/wheelhouse
Processing ./offline_env/wheelhouse/vllm-0.20.1-1-cp38-abi3-manylinux_2_35_x86_64.whl (from -r /kaggle/working/offline_env/reqs.txt (line 1))
Processing ./offline_env/wheelhouse/trl-1.3.0-py3-none-any.whl (from -r /kaggle/working/offline_env/reqs.txt (line 2))
Processing ./offline_env/wheelhouse/transformers-5.8.0-py3-none-any.whl (from -r /kaggle/working/offline_env/reqs.txt (line 3))
Processing ./offline_env/wheelhouse/peft-0.19.1-py3-none-any.whl (from -r /kaggle/working/offline_env/reqs.txt (line 4))
Processing ./offline_env/wheelhouse/accelerate-1.13.0-py3-none-any.whl (from -r /kaggle/working/offline_env/reqs.txt (line 5))
Processing ./offline_env/wheelhouse/qwen_vl_utils-0.0.14-py3-none-any.whl (from -r /kaggle/working/offline_env/reqs.txt (line 6))
Processing ./offline_env/whee

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
rfc3161-client 1.0.5 requires cryptography<47,>=43, but you have cryptography 48.0.0 which is incompatible.
sigstore 4.2.0 requires cryptography<47,>=42, but you have cryptography 48.0.0 which is incompatible.
sigstore 4.2.0 requires rich<15,>=13, but you have rich 15.0.0 which is incompatible.
ydata-profiling 4.18.1 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires 


✅ Installation Complete!

🚀 Success!
Transformers: 5.8.0
Tokenizers: 0.22.2
TRL: 1.3.0


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 ── IMPORTS & PATH INJECTION (With Numpy Bypass)
# ═══════════════════════════════════════════════════════════════════════════════
import os
import sys
import importlib.metadata

# 1. CRITICAL: Inject your offline library path FIRST
LOCAL_LIB = "/kaggle/working/lib"
if LOCAL_LIB not in sys.path:
    sys.path.insert(0, LOCAL_LIB)

# 2. Fix CUDA pathing for bitsandbytes
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# 3. THE BYPASS: Trick Unsloth's Numpy Checker
import numpy as np
try:
    # Find out what version is actually installed in your offline folder
    installed_numpy_version = importlib.metadata.version("numpy")
    # Overwrite the version string of the Kaggle pre-loaded numpy
    np.__version__ = installed_numpy_version
except Exception as e:
    pass

# 4. UNSLOTH MUST BE THE ABSOLUTE FIRST EXTERNAL IMPORT

# 5. Now import the rest of your stack
import torch
import transformers
import tokenizers
import trl
import vllm

print("🚀 Success! The numpy ghost has been bypassed.")
print(f"Transformers: {transformers.__version__} | TRL: {trl.__version__}")

🚀 Success! The numpy ghost has been bypassed.
Transformers: 5.8.0 | TRL: 1.3.0


In [4]:
import os
import sys

# 1. Point to your custom library location
custom_lib = "/kaggle/working/lib"

# 2. Tell the current Python process where to look
if custom_lib not in sys.path:
    sys.path.insert(0, custom_lib)

# 3. Tell all SUBPROCESSES where to look (This is the crucial fix)
os.environ["PYTHONPATH"] = f"{custom_lib}:{os.environ.get('PYTHONPATH', '')}"

# 4. Verify vLLM can be imported now
try:
    import vllm
    print(f"vLLM version: {vllm.__version__} successfully imported from {vllm.__file__}")
except ImportError:
    print("vLLM still not found. Check if /kaggle/working/lib/vllm exists.")

vLLM version: 0.20.1 successfully imported from /kaggle/working/lib/vllm/__init__.py


In [5]:
import gc
import torch

# 1. Delete the LLM engine object
if 'llm' in globals():
    del llm

# 2. Force garbage collection
gc.collect()

# 3. Empty the CUDA cache
torch.cuda.empty_cache()


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STYLE RATIONALE DATASET GENERATOR                                          ║
# ║  Teacher : Qwen2.5-VL-72B-AWQ  (vLLM)                                      ║
# ║  Hardware: NVIDIA H100 80 GB                                                 ║
# ║  Output  : CSV with full XML reasoning rationales for every                  ║
# ║            (style × pair × morph_idx) triple                                 ║
# ║  Resume  : crash-safe — checkpoint JSON skips already-done rows              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 ── OFFLINE INSTALLS  (run once with internet ON, then disable)
# ═══════════════════════════════════════════════════════════════════════════════
# !pip install vllm --quiet
# vLLM bundles transformers, so no separate install needed in most Kaggle envs.
# If qwen_vl_utils is not bundled:
# !pip install qwen-vl-utils --quiet
#\═══════════════════════════════════════════════════════
# CELL 2 ── IMPORTS & GLOBAL CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════
import os, re, csv, json, gc, time, random, textwrap
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Set
import pandas as pd

import torch
from PIL import Image
from vllm import LLM, SamplingParams
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

# ── Paths ────────────────────────────────────────────────────────────────────
# ⚠ Fill in your actual Kaggle dataset input paths
MODEL_PATH        = "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/72b-instruct-awq/2"   # AWQ weights
PART1_ROOT        = "/kaggle/input/datasets/somanshu7527/extracted-strips-part-1"  # ← edit
PART2_ROOT        = "/kaggle/input/datasets/somanshu7527/extracted-strips-part-2"  # ← edit

OUTPUT_CSV        = "/kaggle/working/style_rationale_dataset.csv"
CHECKPOINT_FILE   = "/kaggle/working/gen_checkpoint.json"

# ── Dataset structure constants ───────────────────────────────────────────────
IMAGE_SIZE        = 1024          # resize all images to 1024 × 1024
SOURCE_IDX        = 1             # 0.png  = original source image
FINAL_IDX         = 10            # 10.png = fully stylized (perfect-10 exemplar)
MORPH_RANGE       = list(range(1, 11))      # scores 1 … 10 inclusive
EXEMPLAR_PAIR_IDS = [1, 10]        # pair dirs reserved as ICL exemplars (never in CSV)

ALL_STYLES: Set[str] = {
    "abstract", "acrylic", "charcoal", "cubist", "expressive",
    "gouache", "impressionist", "minimalist", "mosaic",
    "pencil", "pop_art", "watercolor",
}

# ── Generation hyper-parameters ───────────────────────────────────────────────
MAX_NEW_TOKENS  = 1200   # generous headroom for full XML + thinking block
TEMPERATURE     = 0.0    # greedy — deterministic, reproducible rationales
BATCH_SIZE      = 32     # vLLM handles the actual GPU scheduling; adjust if OOM
SEED            = 42

# ── Visual style markers (description ONLY — names intentionally absent) ──────
#    These are fed to the teacher model to tell it what to look for.
#    The style NAME is never included so generated rationales generalize
#    to unseen validation styles during downstream SFT / GRPO.
STYLE_MARKERS: Dict[str, Tuple[str, str, str]] = {
    "abstract":      ("non-representational geometric forms",
                      "bold flat primary color blocks",
                      "elimination of figurative elements"),
    "acrylic":       ("thick impasto surface texture",
                      "sharp-edged opaque brushwork",
                      "high-saturation vivid colors"),
    "charcoal":      ("monochromatic grey tonal gradients",
                      "smudged dusty texture marks",
                      "visible hatching lines"),
    "cubist":        ("simultaneous multiple-viewpoint planes",
                      "fractured interlocking geometric facets",
                      "muted ochre and brown palette"),
    "expressive":    ("raw emotionally distorted brushwork",
                      "unblended aggressive color strokes",
                      "angular forms"),
    "gouache":       ("opaque chalky flat color areas",
                      "matte non-reflective finish",
                      "clean sharp color-edge definition"),
    "impressionist": ("short broken dappled brushstrokes",
                      "chromatic color juxtaposition without blending",
                      "soft atmospheric hazy edges"),
    "minimalist":    ("extreme negative white-space areas",
                      "single dominant restrained color",
                      "reductive simplified forms"),
    "mosaic":        ("discrete tile fragments with visible grout lines",
                      "angular hard-edge color segmentation",
                      "tesserae patterns"),
    "pencil":        ("parallel graphite hatching lines",
                      "linear tonal gradation from light to dark",
                      "visible paper grain texture"),
    "pop_art":       ("flat bold primary poster colors",
                      "Ben-Day halftone dot patterns",
                      "thick black outline contour lines"),
    "watercolor":    ("transparent layered wash glazes",
                      "soft wet-edge bloom blending",
                      "paper granulation in wash areas"),
}

def style_desc(style: str) -> str:
    m = STYLE_MARKERS.get(style, ("distinctive texture", "color handling", "mark-making"))
    return f"{m[0]}, {m[1]}, and {m[2]}"


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 ── PROMPT DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════════════

# ── System prompt for the TEACHER (generation) ───────────────────────────────
#    Key differences from the training system prompt:
#      1. Instructs model to NEVER name or identify the style category.
#      2. Tells model that exact metric scores are pre-determined — its job
#         is to produce analytical visual reasoning that supports them.
GENERATION_SYSTEM_PROMPT = """You are an elite Visual Stylometric Analyst and Aesthetic Evaluator.
Your objective is to rigorously analyse the success of a style transfer operation and produce analytical XML reasoning.

# YOUR PRIMARY CONSTRAINT — CRITICAL:
You must NEVER name, identify, or categorize any artistic movement, medium, genre, or named style in your response.
Describe ONLY what you physically observe: brushwork character, texture surface, color handling, edge quality, tonal range, mark-making method, spatial organization, and surface finish.

# Evaluation Approach:
1. UNDERSTAND: Extract the visual markers defining the target from the STYLE MARKERS provided.
2. CALIBRATE: Study the Exemplar Pairs (both rated 10/10) to establish the visual DNA baseline.
3. EXECUTE: Methodically compare the Query Input → Query Result against those markers.
4. VERIFY: Output analytical scores that accurately reflect the current state of the transformation.

# SCORING PHILOSOPHY — READ CAREFULLY:
You are grading on a STRICT academic scale, not a lenient one.
- Scores of 9–10 are RARE. Reserve them only for results that are essentially indistinguishable from the exemplars. If you can observe ANY deviation, it cannot be a 10. If there is more than one subtle flaw, it cannot be a 9.
- Scores of 7–8 represent GOOD but IMPERFECT work. This should be the most common range for competent transfers.
- Scores of 5–6 represent MEDIOCRE work with clear, visible shortcomings across multiple regions.
- Scores of 1–4 represent FAILURE at various degrees.
- A score of 8 is NOT a default. Do not award 8 unless you have explicitly verified that all markers are strongly present with only minor, isolated gaps.
- NEVER round up. If an image is between two scores, assign the LOWER one.
- Your distribution across a full dataset should approximate: 1–4 (~10%), 5–6 (~20%), 7–8 (~50%), 9–10 (~20%). If you find yourself awarding 9s and 10s more than 20% of the time, you are being too lenient.

# GRADING RUBRIC:

Metric 1 — Style Fidelity (1-10 Scale)
  1 (None):        Zero stylistic transformation; image is identical to the source.
  2 (Trace):       Barely perceptible style hints; only 1 marker faintly visible in isolated pixels.
  3 (Emerging):    Early-stage style signals; 1–2 markers weakly present, structure fully dominant.
  4 (Partial):     Style recognizable but shallow; markers present in some regions, not globally applied.
  5 (Moderate):    Key traits present but applied as a surface filter; lacks depth and full coverage.
  6 (Substantial): Most markers clearly visible; style reads correctly but minor regions are undertransformed.
  7 (Strong):      All 3 markers present and consistent; minor structural or tonal deviations remain.
  8 (High):        Strong faithful transfer; style DNA intact with only subtle, isolated gaps in texture or edge treatment. You must cite the specific gap to justify this score.
  9 (Exceptional): Near-perfect transfer; all markers fully integrated. Only 1 imperceptible flaw allowed. You must explicitly name what that single flaw is, or you cannot assign 9.
  10 (Perfect):    Full faithful transfer; every marker perfectly realized; indistinguishable from exemplar quality. Assign ONLY when the result could be mistaken for a new exemplar image.

Metric 2 — Content Preservation (1-10 Scale)
  1 (Destroyed):   Major objects hallucinated or destroyed; layout completely unrecognizable.
  2 (Severe Loss): Primary subject barely identifiable; heavy structural corruption.
  3 (Major Warp):  Subject recognizable but significant feature bleeding or shape distortion.
  4 (Degraded):    Main subject intact but secondary elements warped or lost.
  5 (Partial):     Subject clear but noticeable warping, feature-bleeding, or edge smearing present.
  6 (Acceptable):  Most content preserved; minor distortions at edges or fine detail areas only.
  7 (Good):        Subject and layout well-preserved; only subtle texture-level alterations.
  8 (Very Good):   All primary and secondary elements intact; micro-level details slightly softened. You must identify which micro-details are softened to justify this score.
  9 (Near-Perfect): Spatial relations and fine details fully intact with at most 1 imperceptible deviation. Explicitly name the deviation or you cannot assign 9.
  10 (Perfect):    All spatial relations, objects, and fine details perfectly preserved; zero content loss. Assign ONLY when no deviation of any kind is detectable.

Metric 3 — Rendering Quality (1-10 Scale)
  1 (Unusable):    Heavy noise, severe banding, pixel damage, or complete incoherence.
  2 (Very Poor):   Widespread artifacts, blocking, or corruption across most of the image.
  3 (Poor):        Significant visible seams, blur halos, or aliasing in large regions.
  4 (Below Average): Noticeable artifacts in multiple areas; transitions inconsistent.
  5 (Moderate):    Coherent overall but with visible seams, blur patches, or mild aliasing.
  6 (Acceptable):  Minor artifacts present but contained; overall image reads cleanly.
  7 (Good):        Clean rendering with only subtle imperfections in fine texture zones.
  8 (Very Good):   Sharp and coherent; textures and color transitions natural with negligible artifacts. You must identify where the negligible artifacts appear to justify this score.
  9 (Excellent):   Near-perfect technical execution; crisp edges, smooth gradients, no visible flaws except under close inspection of one isolated region.
  10 (Exceptional): Perfectly sharp; all textures and transitions fully natural; zero rendering artifacts. Assign ONLY when the image is technically flawless in every region.

Always respond in this exact XML structure:
<Thinking_Process>
Step 1: Style Extraction: [3 visual markers cross-validated from BOTH reference pairs]
Step 2: Analogy Check: [Compare Query Input -> Query Result vs each of the 3 markers. For each marker, state: present/partial/absent and cite the specific region.]
Step 3: Conflict Check: [Note any style/content conflicts or rendering corruptions observed. Identify the single worst region for each metric.]
Step 4: Score Justification: [For each metric, state your score and the specific evidence that prevents a higher score. If assigning 9, name the single flaw. If assigning 10, confirm zero deviations detected.]
</Thinking_Process>
<Quantitative_Metrics>
Style Fidelity: [integer]
Content Preservation: [integer]
Rendering Quality: [integer]
</Quantitative_Metrics>
<Final_Output>
Brief Rationale: [Analytical justification. Max 1000 words. Do NOT name the style.]
</Final_Output>"""


def build_generation_user_content(
    ex_a_src: Image.Image, ex_a_dst: Image.Image,
    ex_b_src: Image.Image, ex_b_dst: Image.Image,
    q_src:    Image.Image, q_res:    Image.Image,
    markers:  str,
    morph_idx: int, # Kept for function signature, but removed from prompt text
) -> List[Dict]:
    
    text = (
        f"STYLE MARKERS: The target visual language requires the following physical attributes:\n"
        f"  • {markers}\n\n"
        f"Exemplar Pair A (Perfect 10/10): <image_1> → <image_2>\n"
        f"Exemplar Pair B (Perfect 10/10): <image_3> → <image_4>\n\n"
        f"Query to Evaluate: <image_5> → <image_6>\n\n"
        f"Evaluate the Query accurately. Generate detailed <Thinking_Process> and <Final_Output> reasoning that "
        f"analytically justifies your Style Fidelity, Content Preservation, and Rendering Quality scores "
        f"based on specific visual evidence you observe. Do NOT name or categorize the artistic style."
    )

    return [
        {"type": "image", "image": ex_a_src},
        {"type": "image", "image": ex_a_dst},
        {"type": "image", "image": ex_b_src},
        {"type": "image", "image": ex_b_dst},
        {"type": "image", "image": q_src},
        {"type": "image", "image": q_res},
        {"type": "text",  "text":  text},
    ]

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 ── vLLM MODEL LOADING  (H100 — fill VRAM to the brim)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"[{datetime.now():%H:%M:%S}] Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
print(f"[{datetime.now():%H:%M:%S}] Initialising vLLM engine …")
llm = LLM(
    model                    = MODEL_PATH,
    quantization             = "awq",
    dtype                    = "float16",
    
    # --- CRITICAL MEMORY ADJUSTMENTS ---
    gpu_memory_utilization   = 0.90,            # Reduced from 0.97 to give 10% room for spikes
    max_num_seqs             = 4,               # Reduced from 16 to lower peak activation memory
    max_num_batched_tokens   = 16384,           # Reduced from 65536 to prevent massive prefill spikes
    
    # --- KEEP THESE FOR EFFICIENCY ---
    kv_cache_dtype           = "auto",           # Keep this! It saves massive amounts of VRAM
    enforce_eager            = True,            # Essential for memory stability on VL models
    max_model_len            = 14336,
    enable_chunked_prefill   = True,
    enable_prefix_caching    = True,
    limit_mm_per_prompt      = {"image": 6},
    mm_processor_kwargs      = {
        "max_pixels": IMAGE_SIZE * IMAGE_SIZE,
        "min_pixels": IMAGE_SIZE * IMAGE_SIZE,
    },
    trust_remote_code        = True,
)

sampling_params = SamplingParams(
    temperature         = 0.0,   # Back to 0.0 for deterministic, high-quality teacher labels
    n                   = 1,     # Just 1 generation per image
    max_tokens          = MAX_NEW_TOKENS,
    skip_special_tokens = True,
)

print(f"[{datetime.now():%H:%M:%S}] vLLM engine ready.\n")



# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 ── DATA DISCOVERY & TRIPLE ENUMERATION (DYNAMIC COMBINATIONS)
# ═══════════════════════════════════════════════════════════════════════════════

TRAIN_STYLES = {'abstract', 'charcoal', 'cubist', 'gouache', 'impressionist', 'minimalist', 'pencil', 'watercolor'}
VAL_STYLES   = {'expressive', 'mosaic'}
TEST_STYLES  = {'acrylic', 'pop_art'}

def get_split(style_name: str) -> str:
    if style_name in TRAIN_STYLES: return 'train'
    if style_name in VAL_STYLES:   return 'val'
    if style_name in TEST_STYLES:  return 'test'
    return 'unknown'

def discover_styles(roots: List[str]) -> Dict[str, Path]:
    found: Dict[str, Path] = {}
    for root in roots:
        p = Path(root)
        if not p.exists():
            print(f"  [WARN] Root not found: {root}")
            continue
        for d in sorted(p.iterdir()):
            if d.is_dir() and d.name in ALL_STYLES:
                found[d.name] = d
    print(f"[DATA] Discovered {len(found)} styles: {sorted(found.keys())}")
    return found

def enumerate_triples(style_paths: Dict[str, Path]) -> List[Dict]:
    records = []
    # Deterministic Random instance for crash-safe resumes
    rng = random.Random(SEED)
    
    # 125 query combinations * 10 morphs = 1250 rows per style.
    # 1250 rows * 6 styles (in PART1) = 7500 rows total.
    QUERIES_PER_STYLE = 110 
    
    for style in sorted(style_paths):
        root = style_paths[style]
        split_name = get_split(style)
        
        # 1. Find all valid subdirectories that contain 1.png through 10.png
        all_dirs = [d for d in root.iterdir() if d.is_dir() and d.name.isdigit()]
        valid_dirs = []
        for d in all_dirs:
            if all((d / f"{i}.png").exists() for i in range(1, 11)):
                valid_dirs.append(d)
                
        # Sort to guarantee deterministic order across OS/Filesystems before sampling
        valid_dirs = sorted(valid_dirs, key=lambda x: int(x.name))
        
        if len(valid_dirs) < 3:
            print(f"  [WARN] Style '{style}' has fewer than 3 valid folders. Skipping.")
            continue
            
        # 2. Dynamically sample unique combinations of (Ex_A, Ex_B, Query)
        # This prevents the script from failing if a folder has fewer than 127 total directories.
        combinations = set()
        attempts = 0
        
        while len(combinations) < QUERIES_PER_STYLE and attempts < (QUERIES_PER_STYLE * 10):
            # Pick 3 unique directories randomly
            sampled_triplet = tuple(rng.sample(valid_dirs, 3))
            combinations.add(sampled_triplet)
            attempts += 1
            
        # 3. Build records for the 10 morph indices of each combination
        for ex_a_dir, ex_b_dir, query_dir in combinations:
            ex_a_src = ex_a_dir / f"{SOURCE_IDX}.png"
            ex_a_dst = ex_a_dir / f"{FINAL_IDX}.png"
            ex_b_src = ex_b_dir / f"{SOURCE_IDX}.png"
            ex_b_dst = ex_b_dir / f"{FINAL_IDX}.png"
            q_src    = query_dir / f"{SOURCE_IDX}.png"
            
            for morph_idx in MORPH_RANGE:
                q_res = query_dir / f"{morph_idx}.png"
                
                records.append({
                    "style":        style,
                    "split":        split_name,
                    "morph_idx":    morph_idx,
                    "pair_dir":     query_dir.name,
                    "ex_a_src_path": str(ex_a_src),
                    "ex_a_dst_path": str(ex_a_dst),
                    "ex_b_src_path": str(ex_b_src),
                    "ex_b_dst_path": str(ex_b_dst),
                    "q_src_path":    str(q_src),
                    "q_res_path":    str(q_res),
                })
                
    # Shuffle so batches mix styles — helps VRAM batch diversity
    rng.shuffle(records)
    print(f"[DATA] Total triples to generate: {len(records):,}")
    return records

# 👇 EXPLICITLY LIMIT TO ONLY PART1_ROOT FOR THIS NOTEBOOK 👇
STYLE_PATHS = discover_styles([PART1_ROOT])
ALL_RECORDS = enumerate_triples(STYLE_PATHS)


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 ── RESUME LOGIC  (skip already-completed rows)
# ═══════════════════════════════════════════════════════════════════════════════

def load_checkpoint() -> Set[Tuple[str, str, int]]:
    """Return set of (style, pair_dir, morph_idx) already written to CSV."""
    done: Set[Tuple[str, str, int]] = set()
    if Path(OUTPUT_CSV).exists():
        with open(OUTPUT_CSV, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    done.add((row["style"], row["pair_dir"], int(row["morph_idx"])))
                except (KeyError, ValueError):
                    pass
        print(f"[RESUME] Found {len(done):,} completed rows in existing CSV.")
    return done


def filter_pending(records: List[Dict], done: Set) -> List[Dict]:
    pending = [r for r in records
               if (r["style"], r["pair_dir"], r["morph_idx"]) not in done]
    print(f"[RESUME] {len(pending):,} rows still pending.")
    return pending


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 ── XML PARSER (FREE EVALUATION + HOLISTIC SCORING)
# ═══════════════════════════════════════════════════════════════════════════════

def calculate_holistic_score(sf, cp, rq):
    weighted = (sf * 0.40) + (cp * 0.30) + (rq * 0.30)
    
    # If the image is mangled or heavily artifacted, it caps the score.
    if rq <= 3 or cp <= 3:
        return min(rq, cp) 
    
    return max(1, min(10, int(round(weighted))))

def parse_xml_response(text: str) -> Dict:
    sf_m  = re.search(r"Style Fidelity\s*:\s*([0-9]+)",       text, re.I)
    cp_m  = re.search(r"Content Preservation\s*:\s*([0-9]+)",  text, re.I)
    rq_m  = re.search(r"Rendering Quality\s*:\s*([0-9]+)",     text, re.I)

    def _clamp(m):
        return max(1, min(10, int(m.group(1)))) if m else None

    sf = _clamp(sf_m)
    cp = _clamp(cp_m)
    rq = _clamp(rq_m)

    # It's a valid generation as long as the VLM gave us all 3 integers
    gen_ok = all(v is not None for v in [sf, cp, rq])
    
    if gen_ok:
        final = calculate_holistic_score(sf, cp, rq)
    else:
        final = None

    return {
        "sf":           sf,
        "cp":           cp,
        "rq":           rq,
        "final_score":  final,
        "generation_ok": gen_ok,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 ── IMAGE LOADER  (cached to avoid re-opening the same exemplar PNGs)
# ═══════════════════════════════════════════════════════════════════════════════

_img_cache: Dict[str, Image.Image] = {}

def load_image_cached(path: str) -> Image.Image:
    if path not in _img_cache:
        _img_cache[path] = (
            Image.open(path).convert("RGB")
                 .resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        )
    return _img_cache[path]

def evict_query_images(record: Dict):
    """
    Exemplar images repeat across thousands of samples — keep them cached.
    Query images are unique per triple — evict after use to cap RAM.
    """
    for key in ("q_src_path", "q_res_path"):
        path = record[key]
        _img_cache.pop(path, None)


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 ── PROMPT BUILDER  (per record → vLLM input dict)
# ═══════════════════════════════════════════════════════════════════════════════

def build_vllm_input(record: Dict) -> Dict:
    """
    Construct the vLLM input dict for one (style, pair, morph) triple.
    Uses cached exemplar images; query images are loaded fresh.
    """
    markers = style_desc(record["style"])

    images = [
        load_image_cached(record["ex_a_src_path"]),
        load_image_cached(record["ex_a_dst_path"]),
        load_image_cached(record["ex_b_src_path"]),
        load_image_cached(record["ex_b_dst_path"]),
        load_image_cached(record["q_src_path"]),
        load_image_cached(record["q_res_path"]),
    ]

    user_content = build_generation_user_content(
        *images,
        markers   = markers,
        morph_idx = record["morph_idx"],
    )

    messages = [
        {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    # Apply Qwen2.5-VL chat template → text with image placeholders
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Extract PIL images in the correct order for vLLM
    image_inputs, _ = process_vision_info(messages)

    return {
        "prompt":           text,
        "multi_modal_data": {"image": image_inputs},
    }


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10 ── CSV WRITER SETUP
# ═══════════════════════════════════════════════════════════════════════════════

CSV_COLUMNS = [
    # ── Identifiers ──────────────────────────────────────────────────────────
    "style",
    "split",           # <--- ADDED SPLIT COLUMN
    "pair_dir",
    "morph_idx",

    # ── Image paths (6 columns) ───────────────────────────────────────────────
    "ex_a_src_path",
    "ex_a_dst_path",
    "ex_b_src_path",
    "ex_b_dst_path",
    "q_src_path",
    "q_res_path",

    # ── Model output ─────────────────────────────────────────────────────────
    "response",        # full raw XML from model

    # ── Parsed scores ─────────────────────────────────────────────────────────
    "sf",              # Style Fidelity (parsed)
    "cp",              # Content Preservation (parsed)
    "rq",              # Rendering Quality (parsed)
    "final_score",     # holistic score applied

    # ── Quality flag ──────────────────────────────────────────────────────────
    "generation_ok",   # True iff all 3 scores parsed
]

csv_file_exists = Path(OUTPUT_CSV).exists()
csv_fh   = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
csv_writer = csv.DictWriter(csv_fh, fieldnames=CSV_COLUMNS, extrasaction="ignore")
if not csv_file_exists:
    csv_writer.writeheader()
    csv_fh.flush()


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11 ── REGENERATION LOOP (re-score rows from an existing CSV)
# ═══════════════════════════════════════════════════════════════════════════════

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_CSV  = "/kaggle/input/datasets/bhavyajain12121/style-rationale-dataset1/style_rationale_dataset/style_rationale_dataset(110).csv"  # ← rows to re-score
OUTPUT_CSV = "/kaggle/working/style_rationale_dataset_(110)_v2.csv"               # ← new output
# ──────────────────────────────────────────────────────────────────────────────

# Load the input CSV as the source of records
input_df = pd.read_csv(INPUT_CSV)
print(f"[REGEN] Loaded {len(input_df):,} rows from {INPUT_CSV}")

# Convert each row back into the record dict format build_vllm_input() expects
RECORD_KEYS = [
    "style", "split", "morph_idx", "pair_dir",
    "ex_a_src_path", "ex_a_dst_path",
    "ex_b_src_path", "ex_b_dst_path",
    "q_src_path",    "q_res_path",
]
all_records = input_df[RECORD_KEYS].to_dict(orient="records")

# Resume logic — skip already re-scored rows in OUTPUT_CSV
def load_regen_checkpoint() -> Set[Tuple[str, str, int]]:
    done: Set[Tuple[str, str, int]] = set()
    if Path(OUTPUT_CSV).exists():
        with open(OUTPUT_CSV, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    done.add((row["style"], row["pair_dir"], int(row["morph_idx"])))
                except (KeyError, ValueError):
                    pass
        print(f"[RESUME] Found {len(done):,} already re-scored rows in output CSV.")
    return done

done_set = load_regen_checkpoint()
pending  = [r for r in all_records
            if (r["style"], r["pair_dir"], int(r["morph_idx"])) not in done_set]
print(f"[REGEN] {len(pending):,} rows pending re-score.")

# CSV writer setup
csv_file_exists = Path(OUTPUT_CSV).exists()
csv_fh     = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
csv_writer = csv.DictWriter(csv_fh, fieldnames=CSV_COLUMNS, extrasaction="ignore")
if not csv_file_exists:
    csv_writer.writeheader()
    csv_fh.flush()

# Main loop
total     = len(pending)
n_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
n_done    = 0
n_ok      = 0
t_start   = time.time()

TIME_LIMIT_SECONDS = 11.5 * 3600

print(f"\n{'═'*70}")
print(f"  Regeneration Run (Updated Prompt) — {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"  Target rows : {total:,}  |  Batch size : {BATCH_SIZE}")
print(f"  Timeout     : 11.5 Hours")
print(f"{'═'*70}\n")

for batch_idx in range(0, total, BATCH_SIZE):
    batch_records = pending[batch_idx : batch_idx + BATCH_SIZE]
    batch_num     = batch_idx // BATCH_SIZE + 1

    vllm_inputs = []
    for rec in batch_records:
        try:
            vllm_inputs.append(build_vllm_input(rec))
        except Exception as e:
            print(f"  [SKIP] Prompt build failed: {e}")
            vllm_inputs.append(None)

    valid_inputs = [(i, rec, inp) for i, (rec, inp)
                    in enumerate(zip(batch_records, vllm_inputs)) if inp is not None]

    t_batch = time.time()
    try:
        raw_outputs = llm.generate([inp for _, _, inp in valid_inputs], sampling_params)
    except Exception as e:
        print(f"  [ERROR] Batch failed: {e}")
        continue

    for (_, rec, _), out in zip(valid_inputs, raw_outputs):
        raw_text = out.outputs[0].text.strip()
        parsed   = parse_xml_response(raw_text)

        row = {
            **rec,
            "response":      raw_text,
            "sf":            parsed["sf"],
            "cp":            parsed["cp"],
            "rq":            parsed["rq"],
            "final_score":   parsed["final_score"],
            "generation_ok": parsed["generation_ok"],
        }
        csv_writer.writerow(row)

        n_done += 1
        if parsed["generation_ok"]:
            n_ok += 1

        evict_query_images(rec)

    csv_fh.flush()

    batch_sec = time.time() - t_batch
    elapsed   = time.time() - t_start
    rate      = n_done / elapsed if elapsed > 0 else 0
    eta_sec   = (total - n_done) / rate if rate > 0 else 0

    print(f"  Batch {batch_num:>4}/{n_batches} | Done {n_done:>6,}/{total:,} | OK: {n_ok} | {batch_sec:.1f}s/batch | ETA {eta_sec/60:.0f}m")

    if elapsed > TIME_LIMIT_SECONDS:
        print(f"\n{'='*70}")
        print(f"  [TIMEOUT] 11.5 hours reached. Saving remaining records...")

        remaining_pending = pending[batch_idx + BATCH_SIZE:]
        remaining_file    = "/kaggle/working/remaining_regen_records.json"
        with open(remaining_file, "w") as f:
            json.dump(remaining_pending, f, indent=4)

        print(f"  Saved {len(remaining_pending):,} remaining rows to {remaining_file}")
        print(f"{'='*70}\n")
        break

csv_fh.close()

print(f"\n{'═'*70}")
print(f"  Regeneration complete/stopped — {datetime.now():%H:%M:%S}")
print(f"  Total processed     : {n_done:,}")
print(f"  Valid Responses (OK): {n_ok:,} ({100*n_ok/n_done if n_done else 0:.1f}%)")
print(f"  Output CSV          : {OUTPUT_CSV}")
print(f"{'═'*70}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12 ── POST-GENERATION VALIDATION & STATISTICS
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd

if Path(OUTPUT_CSV).exists():
    df = pd.read_csv(OUTPUT_CSV)
    
    print(f"\n{'═'*70}")
    print(f"  DATASET VALIDATION REPORT")
    print(f"{'═'*70}")
    print(f"\n  Total rows        : {len(df):,}")
    
    if len(df) > 0:
        print(f"  generation_ok     : {df['generation_ok'].sum():,}  "
              f"({100*df['generation_ok'].mean():.1f}%)")
        print(f"  Null responses    : {df['response'].isna().sum()}")
        
        print(f"\n  Rows per style:")
        style_counts = df.groupby("style")["generation_ok"].agg(["count", "sum"])
        style_counts.columns = ["total", "ok"]
        style_counts["ok_pct"] = (100 * style_counts["ok"] / style_counts["total"]).round(1)
        print(style_counts.to_string())
        
        ok_df = df[df["generation_ok"] == True]
        print(f"\n  Exact match rate  : {len(ok_df):,}/{len(df):,}")
        
        drift_df = df[(df["generation_ok"] == False) & df["sf"].notna()]
        if len(drift_df):
            drift_df = drift_df.copy()
            drift_df["delta"] = (drift_df["final_score"] - drift_df["morph_idx"]).abs()
            print(f"  Score-drift rows MAE : {drift_df['delta'].mean():.3f} pts")
        
        # Crash-proof check for the sample printout
        if not ok_df.empty:
            print(f"\n  Sample response (first generation_ok row):\n")
            sample = ok_df.iloc[0]
            print(f"  Style: {sample['style']}  |  Split: {sample['split']}  |  morph_idx: {sample['morph_idx']}")
            print(f"  SF={sample['sf']}  CP={sample['cp']}  RQ={sample['rq']}  final={sample['final_score']}")
            print("\n  --- Response ---")
            print(textwrap.indent(str(sample["response"])[:1200], "  "))
        else:
            print(f"\n  [INFO] No valid 'generation_ok' rows found yet to display as a sample.")
    else:
        print("\n  [INFO] The CSV file exists but is empty.")
    print(f"{'═'*70}\n")
else:
    print(f"\n[ERROR] Output CSV not found at {OUTPUT_CSV}")

[15:34:14] Loading processor …
[15:34:15] Initialising vLLM engine …


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
[W509 15:34:34.193788652 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
(EngineCore pid=260) 2026-05-09 15:35:57,001 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=260) 2026-05-09 15:35:58,554 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends


[15:36:00] vLLM engine ready.

[DATA] Discovered 6 styles: ['abstract', 'acrylic', 'charcoal', 'cubist', 'expressive', 'minimalist']
[DATA] Total triples to generate: 6,600
[REGEN] Loaded 3,520 rows from /kaggle/input/datasets/bhavyajain12121/style-rationale-dataset1/style_rationale_dataset/style_rationale_dataset(110).csv
[REGEN] 3,520 rows pending re-score.

══════════════════════════════════════════════════════════════════════
  Regeneration Run (Updated Prompt) — 2026-05-09 15:36:16
  Target rows : 3,520  |  Batch size : 32
  Timeout     : 11.5 Hours
══════════════════════════════════════════════════════════════════════



Rendering prompts:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Batch  109/110 | Done  3,488/3,520 | OK: 3488 | 344.1s/batch | ETA 6m


Rendering prompts:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Batch  110/110 | Done  3,520/3,520 | OK: 3520 | 373.8s/batch | ETA 0m

══════════════════════════════════════════════════════════════════════
  Regeneration complete/stopped — 02:52:06
  Total processed     : 3,520
  Valid Responses (OK): 3,520 (100.0%)
  Output CSV          : /kaggle/working/style_rationale_dataset_(110)_v2.csv
══════════════════════════════════════════════════════════════════════


══════════════════════════════════════════════════════════════════════
  DATASET VALIDATION REPORT
══════════════════════════════════════════════════════════════════════

  Total rows        : 3,520
  generation_ok     : 3,520  (100.0%)
  Null responses    : 0

  Rows per style:
               total   ok  ok_pct
style                            
gouache          602  602   100.0
impressionist    597  597   100.0
mosaic           571  571   100.0
pencil           588  588   100.0
pop_art          565  565   100.0
watercolor       597  597   100.0

  Exact match rate  : 3,520/3,520

  Samp